In [1]:
import pandas as pd
from pathlib import Path
import glob
import duckdb
import gc
import pyarrow.parquet as pq

In [2]:
folder = Path.home() / 'Documents' / 'moltbook' / 'arctic_shift'

def carica_jsonl(pattern):
    dfs = []
    for f in sorted(folder.glob(pattern)):
        df = pd.read_json(f, lines=True)
        if df.empty:
            print(f"Vuoto, salto: {f.name}")
            continue
        df['subreddit_file'] = f.stem.removeprefix('r_').rsplit('_', 1)[0]
        dfs.append(df)
        print(f"{f.name}: {len(df)} righe")
    return pd.concat(dfs, ignore_index=True)

In [3]:
df_reddit_comments = carica_jsonl('r_*_comments.jsonl')

r_philosophy_comments.jsonl: 21318 righe
r_physics_comments.jsonl: 56326 righe


In [4]:
df_reddit_posts    = carica_jsonl('r_*_posts.jsonl')

r_philosophy_posts.jsonl: 2376 righe
r_physics_posts.jsonl: 4298 righe


In [5]:
df_reddit_posts["created_utc"] = pd.to_datetime(df_reddit_posts["created_utc"], unit="s", utc=True) 
df_reddit_posts = df_reddit_posts[['author','downs','id','name','num_comments','created_utc','subreddit','ups']]

In [7]:
# convert to datetime
df_reddit_comments["created_utc"] = pd.to_datetime(df_reddit_comments["created_utc"], unit="s", utc=True)

# remove prefix in link_id and parent_id
df = df_reddit_comments
p = df["parent_id"]

# tutti i parent_id senza prefisso (tengo anche l'originale)
df["parent_id_raw"]   = p
df["parent_id_clean"] = p.str.replace(r"^t[0-9]_", "", regex=True)

# solo quelli che puntano a un post (t3_)
df["parent_post_id"]    = p.where(p.str.startswith("t3_", na=False)).str[3:]

# solo quelli che puntano a un commento (t1_)
df["parent_comment_id"] = p.where(p.str.startswith("t1_", na=False)).str[3:]

# link_id (il post della discussione) senza prefisso
df["link_id"] = df["link_id"].str.replace(r"^t[0-9]_", "", regex=True)
    
df_reddit_comments = df_reddit_comments[['author','created_utc','downs','id','link_id','parent_id_clean','parent_post_id','parent_comment_id','subreddit','ups']]

In [8]:
df_reddit_comments['interaction_type'] = 'comment'
df_reddit_posts['interaction_type'] = 'post'

In [9]:
# rinomina colonne per uniformare dataset reddit e moltbook
df_reddit_posts = df_reddit_posts.rename(columns={'author':'agent_id'})
df_reddit_comments = df_reddit_comments.rename(columns={'author':'agent_id'})
df_reddit_posts = df_reddit_posts.rename(columns={'created_utc':'created_at'})
df_reddit_comments = df_reddit_comments.rename(columns={'created_utc':'created_at'})
df_reddit_posts = df_reddit_posts.rename(columns={'subreddit':'submolt'})
df_reddit_comments = df_reddit_comments.rename(columns={'subreddit':'submolt'})
df_reddit_comments = df_reddit_comments.rename(columns={'link_id':'post_id'})
df_reddit_comments = df_reddit_comments.rename(columns={'parent_id_clean':'parent_id'})

In [12]:
# elimino commenti pubblicati su post che non appartengono alla finestra temporale selezionata
df_reddit_comments = df_reddit_comments[df_reddit_comments['post_id'].isin(df_reddit_posts['id'])]

In [13]:
df_reddit_comments.to_pickle('df_reddit_comments.pkl') 
df_reddit_posts.to_pickle('df_reddit_posts.pkl')

In [27]:
df_reddit_comments.head()

,agent_id,created_at,downs,id,post_id,parent_id,parent_post_id,parent_comment_id,submolt,ups,interaction_type
1,AutoModerator,2026-05-01 00:35:58+00:00,0,oj8dtp4,1t0e3b6,1t0e3b6,1t0e3b6,NaN,philosophy,1,comment
26,AutoModerator,2026-05-01 01:41:07+00:00,0,oj8ovv5,1t0fifb,1t0fifb,1t0fifb,NaN,philosophy,1,comment
32,revereddesecration,2026-05-01 01:46:35+00:00,0,oj8psi9,1t0fifb,1t0fifb,1t0fifb,NaN,philosophy,37,comment
33,theperipherypeople,2026-05-01 01:47:16+00:00,0,oj8pwod,1t0fifb,1t0fifb,1t0fifb,NaN,philosophy,27,comment
36,TheVideoGameCritic,2026-05-01 01:52:11+00:00,0,oj8qqwm,1t0fifb,oj8pwod,NaN,oj8pwod,philosophy,7,comment


In [28]:
df_reddit_posts.head()

,agent_id,downs,id,name,num_comments,created_at,submolt,ups,interaction_type
0,PuzzleheadedSport281,0,1t0e3b6,t3_1t0e3b6,1,2026-05-01 00:35:57+00:00,philosophy,1,post
1,OkReport5065,0,1t0fifb,t3_1t0fifb,25,2026-05-01 01:41:06+00:00,philosophy,0,post
2,Silientium,0,1t0h8cx,t3_1t0h8cx,1,2026-05-01 02:59:12+00:00,philosophy,1,post
3,Silientium,0,1t0huhx,t3_1t0huhx,1,2026-05-01 03:28:32+00:00,philosophy,1,post
4,amsirm1,0,1t0j9ye,t3_1t0j9ye,1,2026-05-01 04:40:20+00:00,philosophy,1,post


# Moltbook

In [14]:
posts_moltbook = Path.home() / 'Documents' / 'Archive_data' / 'moltbook_posts' / 'data' / 'posts'

duckdb.sql(f"""
    COPY (
        SELECT id, agent_id, submolt, comment_count, created_at, dump_date
        FROM read_parquet('{posts_moltbook}/*.parquet', union_by_name=true)
        QUALIFY row_number() OVER (PARTITION BY id ORDER BY dump_date DESC) = 1
    ) TO 'posts_ridotto_1.parquet' (FORMAT parquet)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [15]:
comments_moltbook = Path.home() / 'Documents' / 'Archive_data' / 'moltbook_comments' / 'data' / 'comments'

duckdb.sql(f"""
    COPY (
        SELECT id, post_id, agent_name, agent_id, parent_id, score, created_at, dump_date
        FROM read_parquet('{comments_moltbook}/*.parquet', union_by_name=true)
        QUALIFY row_number() OVER (PARTITION BY id ORDER BY dump_date DESC) = 1
    ) TO 'comments_ridotto_1.parquet' (FORMAT parquet)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [16]:
df_moltbook_posts = pd.read_parquet("posts_ridotto_1.parquet", dtype_backend='pyarrow')

In [17]:
df_moltbook_comments = pd.read_parquet("comments_ridotto_1.parquet", dtype_backend='pyarrow')

In [20]:
# aggiungo colonna submolt ai commenti
ids_usati = df_moltbook_comments["post_id"].dropna().unique()

ps = (df_moltbook_posts.loc[df_moltbook_posts["id"].isin(ids_usati), ["id", "submolt"]]
      .drop_duplicates("id"))
mappa = ps.set_index("id")["submolt"]
del ps; gc.collect()

df_moltbook_comments["submolt"] = (df_moltbook_comments["post_id"]
                                   .map(mappa)
                                   .astype("category")) 

In [23]:
# seleziono submolt e finestra temporale
SUB = ["philosophy", "crypto", "technology"]
da, a = pd.Timestamp("2026-05-01", tz="UTC"), pd.Timestamp("2026-10-01", tz="UTC")

df_moltbook_comments_small = df_moltbook_comments[df_moltbook_comments["submolt"].isin(SUB) & (df_moltbook_comments["created_at"] >= da) & (df_moltbook_comments["created_at"] < a)].copy()
df_moltbook_posts_small = df_moltbook_posts[df_moltbook_posts["submolt"].isin(SUB) & (df_moltbook_posts["created_at"] >= da) & (df_moltbook_posts["created_at"] < a)].copy()

df_moltbook_comments_small['interaction_type'] = 'comment'
df_moltbook_posts_small['interaction_type'] = 'post'

In [26]:
df_moltbook_comments_small.to_pickle('df_moltbook_comments_small.pkl')
df_moltbook_posts_small.to_pickle('df_moltbook_posts_small.pkl')